# Лабораторна робота №4. Візуалізація даних 2 (Інтерактивна гармоніка)

**Студент:** Процан Артем Вікторович (ФБ-46)
**Мета:** Створення інтерактивного графіка функції гармоніки з накладеним шумом, фільтрація сигналу та реалізація GUI за допомогою `matplotlib.widgets`.

## Інструкція для користувача:
1. Запустіть комірку з кодом нижче. Графік з'явиться у новому вікні.
2. Використовуйте слайдери, щоб змінювати амплітуду, частоту та фазу гармоніки.
3. Змінюйте параметри шуму для генерування нових випадкових значень.
4. Використовуйте слайдер фільтра для налаштування частоти зрізу.
5. За допомогою чекбоксів можна вмикати/вимикати відображення ліній.
6. Кнопка Reset повертає всі параметри до початкового стану.

In [3]:
%matplotlib tk

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import iirfilter, filtfilt

def harmonic_with_noise(t, amplitude, frequency, phase, noise_mean, noise_covariance, current_noise=None):
    clean_harmonic = amplitude * np.sin(2 * np.pi * frequency * t + phase)
    
    if current_noise is None:
        std_dev = np.sqrt(max(noise_covariance, 0.0))
        current_noise = np.random.normal(noise_mean, std_dev, len(t))
        
    noisy_harmonic = clean_harmonic + current_noise
    return clean_harmonic, noisy_harmonic, current_noise

class InteractiveHarmonic:
    def __init__(self):
        self.t = np.linspace(0, 5, 1000)
        self.init_amp = 1.0
        self.init_freq = 1.0
        self.init_phase = 0.0
        self.init_mean = 0.0
        self.init_cov = 0.1
        self.init_cutoff = 5.0

        self.last_mean = self.init_mean
        self.last_cov = self.init_cov
        self.current_noise = None

        self.clean, self.noisy, self.current_noise = harmonic_with_noise(
            self.t, self.init_amp, self.init_freq, self.init_phase, 
            self.init_mean, self.init_cov
        )
        self.filtered = self.apply_filter(self.noisy, self.init_cutoff)

        self.fig, self.ax = plt.subplots(figsize=(10, 8))
        plt.subplots_adjust(left=0.1, bottom=0.5)
        
        self.ax.set_title('Інтерактивна гармоніка з фільтрацією', fontsize=14)
        self.ax.set_xlabel('Час (t)')
        self.ax.set_ylabel('Амплітуда y(t)')
        self.ax.grid(True)

        self.line_noisy, = self.ax.plot(self.t, self.noisy, label='Зашумлена', color='red', alpha=0.4)
        self.line_filtered, = self.ax.plot(self.t, self.filtered, label='Відфільтрована', color='blue', lw=2)
        self.line_clean, = self.ax.plot(self.t, self.clean, label='Чиста', color='black', lw=2, linestyle='--')
        
        self.ax.legend(loc='upper right')

        axcolor = 'lightgoldenrodyellow'
        
        self.ax_amp = plt.axes([0.15, 0.40, 0.65, 0.03], facecolor=axcolor)
        self.ax_freq = plt.axes([0.15, 0.35, 0.65, 0.03], facecolor=axcolor)
        self.ax_phase = plt.axes([0.15, 0.30, 0.65, 0.03], facecolor=axcolor)
        
        self.s_amp = Slider(self.ax_amp, 'Амплітуда', 0.1, 5.0, valinit=self.init_amp)
        self.s_freq = Slider(self.ax_freq, 'Частота', 0.1, 5.0, valinit=self.init_freq)
        self.s_phase = Slider(self.ax_phase, 'Фаза', 0.0, 2*np.pi, valinit=self.init_phase)

        self.ax_mean = plt.axes([0.15, 0.22, 0.65, 0.03], facecolor='lightgrey')
        self.ax_cov = plt.axes([0.15, 0.17, 0.65, 0.03], facecolor='lightgrey')
        
        self.s_mean = Slider(self.ax_mean, 'Шум (Середнє)', -2.0, 2.0, valinit=self.init_mean)
        self.s_cov = Slider(self.ax_cov, 'Шум (Дисперсія)', 0.0, 2.0, valinit=self.init_cov)

        self.ax_cutoff = plt.axes([0.15, 0.09, 0.65, 0.03], facecolor='lightblue')
        self.s_cutoff = Slider(self.ax_cutoff, 'Фільтр (Зріз)', 0.1, 20.0, valinit=self.init_cutoff)

        self.ax_check = plt.axes([0.82, 0.15, 0.15, 0.15])
        self.check = CheckButtons(self.ax_check, ['Шум', 'Фільтр', 'Чиста'], [True, True, True])

        self.ax_reset = plt.axes([0.82, 0.05, 0.1, 0.04])
        self.b_reset = Button(self.ax_reset, 'Reset', hovercolor='0.975')

        self.s_amp.on_changed(self.update)
        self.s_freq.on_changed(self.update)
        self.s_phase.on_changed(self.update)
        self.s_mean.on_changed(self.update)
        self.s_cov.on_changed(self.update)
        self.s_cutoff.on_changed(self.update)
        self.check.on_clicked(self.toggle_visibility)
        self.b_reset.on_clicked(self.reset)

    def apply_filter(self, data, cutoff):
        fs = 200.0 
        nyq = 0.5 * fs
        normal_cutoff = cutoff / nyq
        normal_cutoff = np.clip(normal_cutoff, 0.01, 0.99)
        
        b, a = iirfilter(3, normal_cutoff, btype='low', ftype='butter')
        return filtfilt(b, a, data)

    def update(self, val):
        amp = self.s_amp.val
        freq = self.s_freq.val
        phase = self.s_phase.val
        mean = self.s_mean.val
        cov = self.s_cov.val
        cutoff = self.s_cutoff.val

        if mean != self.last_mean or cov != self.last_cov:
            self.clean, self.noisy, self.current_noise = harmonic_with_noise(
                self.t, amp, freq, phase, mean, cov, current_noise=None
            )
            self.last_mean = mean
            self.last_cov = cov
        else:
            self.clean, self.noisy, _ = harmonic_with_noise(
                self.t, amp, freq, phase, mean, cov, current_noise=self.current_noise
            )

        self.filtered = self.apply_filter(self.noisy, cutoff)

        self.line_clean.set_ydata(self.clean)
        self.line_noisy.set_ydata(self.noisy)
        self.line_filtered.set_ydata(self.filtered)
        
        self.ax.relim()
        self.ax.autoscale_view()
        self.fig.canvas.draw_idle()

    def toggle_visibility(self, label):
        flags = self.check.get_status()
        self.line_noisy.set_visible(flags[0])
        self.line_filtered.set_visible(flags[1])
        self.line_clean.set_visible(flags[2])
        self.fig.canvas.draw_idle()

    def reset(self, event):
        self.s_amp.reset()
        self.s_freq.reset()
        self.s_phase.reset()
        self.s_mean.reset()
        self.s_cov.reset()
        self.s_cutoff.reset()

app = InteractiveHarmonic()
plt.show(block=True)